In [15]:
import pandas as pd

In [16]:
cleanedData = pd.read_csv("student_productivity_distraction_dataset_20000.csv")

In [17]:
cleanedData.head()

,student_id,age,gender,study_hours_per_day,sleep_hours,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,exercise_minutes,assignments_completed,attendance_percentage,stress_level,focus_score,final_grade,productivity_score
0,1,23,Female,4.35,3.63,3.38,2.73,1.83,5.26,6,347,111,2,57.21,10,57,81.87,33.78
1,2,20,Male,6.14,6.58,5.48,1.51,3.13,1.73,13,403,28,10,91.27,10,49,60.90,48.99
2,3,29,Female,4.98,3.26,4.83,3.63,0.18,4.71,1,419,102,8,63.14,2,38,86.22,36.60
3,4,27,Female,3.19,4.58,10.06,3.95,5.75,2.52,9,178,28,18,40.51,6,50,71.77,19.87
4,5,24,Male,7.67,6.21,3.02,1.59,5.46,5.65,8,436,105,7,45.53,6,41,90.13,52.90


In [18]:
cleanedData.columns

Index(['student_id', 'age', 'gender', 'study_hours_per_day', 'sleep_hours',
       'phone_usage_hours', 'social_media_hours', 'youtube_hours',
       'gaming_hours', 'breaks_per_day', 'coffee_intake_mg',
       'exercise_minutes', 'assignments_completed', 'attendance_percentage',
       'stress_level', 'focus_score', 'final_grade', 'productivity_score'],
      dtype='str')

Soriting the columns I need

In [19]:
columns_needed = [
    "study_hours_per_day",
    "sleep_hours",
    "exercise_minutes",
    "stress_level",
    "final_grade"
]

In [20]:
needed_data = cleanedData[columns_needed]
needed_data.describe()

,study_hours_per_day,sleep_hours,exercise_minutes,stress_level,final_grade
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,5.254562,6.517799,59.648050,5.478750,70.266409
std,2.742876,2.029784,34.611751,2.866943,17.282277
min,0.500000,3.000000,0.000000,1.000000,40.000000
25%,2.900000,4.770000,30.000000,3.000000,55.460000
50%,5.250000,6.510000,60.000000,5.000000,70.315000
75%,7.640000,8.310000,90.000000,8.000000,85.340000
max,10.000000,10.000000,119.000000,10.000000,99.990000


Making sure there are no missing values

In [21]:
needed_data.isnull().sum()

study_hours_per_day    0
sleep_hours            0
exercise_minutes       0
stress_level           0
final_grade            0
dtype: int64

As the data is complete I can now construct the wellbeing index

Normalising values

In [22]:
needed_data['exercise_norm'] = (needed_data['exercise_minutes'] - needed_data['exercise_minutes'].min()) / \
                                (needed_data['exercise_minutes'].max() - needed_data['exercise_minutes'].min())

needed_data['sleep_norm'] = (needed_data['sleep_hours'] - needed_data['sleep_hours'].min()) / \
                             (needed_data['sleep_hours'].max() - needed_data['sleep_hours'].min())


Stress is inverted as it can be seen as something that would affect the wellbeing of an individual negativley:

In [23]:
needed_data['stress_norm'] = 1 - (needed_data['stress_level'] - needed_data['stress_level'].min()) / \
                                  (needed_data['stress_level'].max() - needed_data['stress_level'].min())

In [24]:
needed_data['wellbeing_score'] = (
    needed_data['exercise_norm'] +
    needed_data['sleep_norm'] +
    needed_data['stress_norm']
) / 3

Addidng the wellbeing score to the data set

In [25]:
cleanedData['wellbeing_score'] = needed_data['wellbeing_score']

In [26]:
needed_data.head(10)

,study_hours_per_day,sleep_hours,exercise_minutes,stress_level,final_grade,exercise_norm,sleep_norm,stress_norm,wellbeing_score
0,4.35,3.63,111,10,81.87,0.932773,0.090000,0.000000,0.340924
1,6.14,6.58,28,10,60.90,0.235294,0.511429,0.000000,0.248908
2,4.98,3.26,102,2,86.22,0.857143,0.037143,0.888889,0.594392
3,3.19,4.58,28,6,71.77,0.235294,0.225714,0.444444,0.301818
4,7.67,6.21,105,6,90.13,0.882353,0.458571,0.444444,0.595123
5,7.18,3.52,12,10,59.48,0.100840,0.074286,0.000000,0.058375
6,9.06,6.36,28,8,62.71,0.235294,0.480000,0.222222,0.312505
7,6.37,4.86,103,6,52.22,0.865546,0.265714,0.444444,0.525235
8,4.19,4.87,42,3,76.15,0.352941,0.267143,0.777778,0.465954
9,7.28,9.56,107,10,88.53,0.899160,0.937143,0.000000,0.612101


Making sure all values are between 0-1

In [27]:
needed_data['wellbeing_score'].describe()

count    20000.000000
mean         0.502049
std          0.173043
min          0.005238
25%          0.380437
50%          0.501872
75%          0.624452
max          0.994818
Name: wellbeing_score, dtype: float64

Now that the wellbeing score is calculated & added to the data det I can calculte the effect of hours studied and a students wellbeing on their final grade. To do so I will use the Pearson Correlation formula

In [29]:
from scipy import stats

Pearson Correlation: Study Hours vs Final Grade

In [30]:
corr_study, p_study = stats.pearsonr(cleanedData['study_hours_per_day'], cleanedData['final_grade'])

print(f"Study Hours vs Final Grade")
print(f"Correlation: {corr_study:.4f}")
print(f"P-value:     {p_study:.4f}")

Study Hours vs Final Grade
Correlation: -0.0122
P-value:     0.0843


Pearson Correlation: Wellbeing Score vs Final Grade

In [31]:
corr_wellbeing, p_wellbeing = stats.pearsonr(cleanedData['wellbeing_score'], cleanedData['final_grade'])

print(f"Wellbeing Score vs Final Grade")
print(f"Correlation: {corr_wellbeing:.4f}")
print(f"P-value:     {p_wellbeing:.4f}")

Wellbeing Score vs Final Grade
Correlation: 0.0119
P-value:     0.0928


Side by side comparison

In [32]:
print("=" * 40)
print(f"{'Predictor':<25} {'Correlation':>11} {'P-value':>8}")
print("=" * 40)
print(f"{'study_hours_per_day':<25} {corr_study:>11.4f} {p_study:>8.4f}")
print(f"{'wellbeing_score':<25} {corr_wellbeing:>11.4f} {p_wellbeing:>8.4f}")
print("=" * 40)

Predictor                 Correlation  P-value
study_hours_per_day           -0.0122   0.0843
wellbeing_score                0.0119   0.0928


Wellbeing seems to have a larger influence on the final grade than hours studied does!

That being said both seem to be close to zero.. Therefore I'll check whether any value seems to directly affect the final grade

In [33]:
all_correlations = cleanedData.corr(numeric_only=True)['final_grade'].sort_values(ascending=False)
print(all_correlations)


final_grade              1.000000
coffee_intake_mg         0.015326
wellbeing_score          0.011884
sleep_hours              0.010543
student_id               0.007043
focus_score              0.004630
assignments_completed    0.003894
gaming_hours             0.002544
productivity_score       0.001954
youtube_hours            0.001579
social_media_hours      -0.000117
age                     -0.000857
exercise_minutes        -0.002677
breaks_per_day          -0.003690
attendance_percentage   -0.005270
phone_usage_hours       -0.012136
study_hours_per_day     -0.012208
stress_level            -0.012216
Name: final_grade, dtype: float64


Since all values seem close to zero, meaning the correlation between them and the final grade is almost none, I have to assume that the data set is sythetically generated.